# 02. Sorted squares

**Tier:** Deep. **Why you'd use this:** Sort a copy, then fix one value and search the rest with two pointers for a pair that makes the total zero. Skip a repeated fixed value so the same triplet is not recorded twice.

**After this lesson you can:** fill a result from both ends of a sorted list; place squares in order in one pass.
About 20 minutes. **Uses:** [Sorted pair](01-sorted-pair-lesson.ipynb)


## Quick review

1. Where do the largest squares sit in a sorted list that may include negatives?
2. Why fill the result from the right end?

<details><summary>Answers</summary>

For 02 sorted squares lesson: skim [Sorted pair](01-sorted-pair-lesson.ipynb) if needed.
</details>


## Recognize it

The list is sorted and may hold negatives. You need the squares in sorted order. The prompt talks about filling from both ends, or about the largest remaining square living at one of the two ends.

Signals: sorted input that is not centered at zero, and an output that must stay sorted after a monotonic transform like squaring. A look-alike that is not this pattern: squaring then calling `sort` — that works, but it spends an extra `n log n` and is not the two-pointer fill.


## Why it works this way

Squaring folds the number line. Large negatives and large positives both become large positives. In a sorted list the candidates for the largest remaining square are only the two ends: `nums[left]` and `nums[right]`. Compare their absolute values, write the larger square at the next open slot from the back of the answer, and move that end inward. The next largest square is again at one of the two new ends. After `n` writes the answer is filled from largest to smallest on the right side, which is sorted order overall.

A square-then-sort pass is simpler to say and costs roughly `n log n` plus the square work. The two-pointer fill does one compare and one write per index, so time stays proportional to `n` after you already have sorted input. Extra memory is the answer list. Overwriting the input in place is possible in some languages; here a fresh list keeps the caller's order intact and matches how Python callers usually expect a new result.

The invariant: every index outside `[left, right]` already has its final square placed, and those placed values are in non-increasing order from the right. That is why writing from the back is required — you always place the next largest remaining square. Writing from the front would need the smallest square first, which sits somewhere near zero, not at a known end. Searching for zero with a third pointer would only complicate a problem the two ends already solve.

This is still two pointers, but the pointers cooperate to build an array rather than to find a pair. The sorted-pair lesson moves ends based on a sum. Here the ends move based on magnitude. Both rely on sorted order making extreme values sit at the edges of the remaining slice.


## When to use it

Use end-to-end fill when a sorted array maps through a function that is large at the extremes (squares, or merging two sorted lists into one). Use square-then-sort when `n` is small and clarity beats the linear pass. Use a heap only when the input streams or is not sorted.

If the transform is not monotone at the ends — for example an arbitrary map that peaks in the middle — the two-end argument collapses and you should sort after transforming instead of forcing this fill.


## Brute force, and why it's slow

Square every value, then sort. Count the sort work by timing the obvious version on growing inputs.


In [1]:
import time


def sorted_squares_slow(nums):
    return sorted(value * value for value in nums)


for size in (2_000, 8_000, 20_000):
    data = list(range(-size // 2, size // 2))
    start = time.perf_counter()
    sorted_squares_slow(data)
    print(size, round(time.perf_counter() - start, 4))

2000 0.0002
8000 0.001
20000 0.001


## Key insight

After squaring, the largest remaining value is always at one of the two ends of the still-unplaced slice. Write that square from the back of the answer and shrink that end.


## Worked example

Return the squares of a sorted list, still sorted. Fill the result from both ends in one pass.


In [2]:
def sorted_squares(nums: list[int]) -> list[int]:
    # 1. Answer slots fill from the largest square backward.
    answer = [0] * len(nums)
    left, right = 0, len(nums) - 1
    for index in range(len(nums) - 1, -1, -1):
        # 2. The next largest square sits at one of the two ends.
        if abs(nums[left]) > abs(nums[right]):
            answer[index] = nums[left] * nums[left]
            left += 1
        else:
            answer[index] = nums[right] * nums[right]
            right -= 1
    return answer


print(sorted_squares([-4, -1, 0, 3, 10]))
print(sorted_squares([-7, -3, 2, 3, 11]))

[0, 1, 9, 16, 100]
[4, 9, 9, 49, 121]


## Trace

`[-4, -1, 0, 3, 10]`:

| write at | compare `|−4|` vs `|10|` | place | left, right |
|---|---|---|---|
| 4 | 4 < 10 | 100 | 0, 3 |
| 3 | 4 > 3 | 16 | 1, 3 |
| 2 | 1 < 3 | 9 | 1, 2 |
| 1 | 1 < 0? no, `1 > 0` | 1 | 2, 2 |
| 0 | ends meet on 0 | 0 | done |

Result: `[0, 1, 9, 16, 100]`.


## Cost

One write per index: time grows with `n`. The answer list uses `n` slots. Square-then-sort adds a comparison sort on top of the same `n` squares, which dominates as `n` grows. Extra pointers are two integers; no hash table sits beside the answer.


## Variants

Not solved here: [3Sum](../../challenges/problems/04-3sum.ipynb), [Container With Most Water](../../challenges/problems/05-container-with-most-water.ipynb).


## Common mistake

This bug is silent. The code compares the raw values instead of their absolute values, so a large negative loses to a smaller positive and the squares come out unsorted. There is no traceback. The print starts with `16`, and the sorted answer starts with `0`.


In [3]:
def sorted_squares_wrong(nums):
    answer = [0] * len(nums)
    left, right = 0, len(nums) - 1
    for index in range(len(nums) - 1, -1, -1):
        if nums[left] > nums[right]:  # ignores sign
            answer[index] = nums[left] * nums[left]
            left += 1
        else:
            answer[index] = nums[right] * nums[right]
            right -= 1
    return answer


print(sorted_squares_wrong([-4, -1, 0, 3, 10]))

[16, 1, 0, 9, 100]


The fix compares `abs` on both ends. Magnitude, not signed order, picks the next square. The buggy cell prints `[16, 1, 0, 9, 100]` for `[-4, -1, 0, 3, 10]` because it compares signed values.


In [4]:
print(sorted_squares([-4, -1, 0, 3, 10]))

[0, 1, 9, 16, 100]


## Python idioms

`abs` decides which end wins. `range(len(nums) - 1, -1, -1)` walks write indexes from the back without a second counter.


In [5]:
def sorted_squares_zip(nums: list[int]) -> list[int]:
    answer = [0] * len(nums)
    left, right = 0, len(nums) - 1
    write = len(nums) - 1
    while left <= right:
        if abs(nums[left]) > abs(nums[right]):
            answer[write] = nums[left] * nums[left]
            left += 1
        else:
            answer[write] = nums[right] * nums[right]
            right -= 1
        write -= 1
    return answer


print(sorted_squares_zip([-4, -1, 0, 3, 10]))

[0, 1, 9, 16, 100]


## Recap

- After squaring, the largest value is at one of the two ends.
- Fill the answer from the back.
- Compare absolute values, not signed order.

Next: [practice](02-sorted-squares-practice.ipynb).
